### **A Simple Data Cleaning Project with NumPy Using a Loan Dataset**
#### **Context:** <br> I am working as a Data Analyst in a Data Science team of a central bank. The team has been assigned to create a credit risk model, <br> which estimates the probability of default for every personal account, I have been tasked as the Data Analyst of the team to take <br> the raw  dataset and prepare it for the models they plan to run. The loan data is a sample from a larger dataset that belongs <br> an affiliate bank in the United Satates, so all the values are in dollars ($), There is the need to provide the euro equivalence. <br> Every categorical variable must be quantified. When measuring creditworthiness, it crucial to be extremely risk-averse and <br> distrustful of any unavailable data - the concensus of the team is that missing information suggests foul play - if the <br> information is unavailable we will just assume the worst. What is worst varies from one column to the next. <br> So, the team has provided us with casting directions fro each variable on the dataset. <br> The loan information is stored in a csv file called "loan-data.csv"

#### Importing Packages (This is a Numpy Project)

In [187]:
import numpy as np

## Setting print options to numpy's regulate and enhance output display 
np.set_printoptions(suppress = True, linewidth = 100, precision = 2)

#### Importing and Loading Dataset

In [188]:
raw_data = np.genfromtxt("loan-data.csv", 
                         delimiter = ';',
                         skip_header = 1,
                         autostrip = True,
                         encoding = 'latin-1')
print(raw_data)

[[48010226.           nan    35000.   ...         nan         nan     9452.96]
 [57693261.           nan    30000.   ...         nan         nan     4679.7 ]
 [59432726.           nan    15000.   ...         nan         nan     1969.83]
 ...
 [50415990.           nan    10000.   ...         nan         nan     2185.64]
 [46154151.           nan         nan ...         nan         nan     3199.4 ]
 [66055249.           nan    10000.   ...         nan         nan      301.9 ]]


#### Handling missing values

In [189]:
print(f"Total number of NaN values: {np.isnan(raw_data).sum()}\n")

Total number of NaN values: 88005



In [190]:
temp_fill = np.nanmax(raw_data) + 1 # generating a temporary fill value
temp_mean = np.nanmean(raw_data, axis = 0) # genrating a temporary column mean

C:\Users\Bhrajo\AppData\Local\Temp\ipykernel_12628\3623578383.py:2: RuntimeWarning: Mean of empty slice
  temp_mean = np.nanmean(raw_data, axis = 0) # genrating a temporary column mean


In [191]:
## Responding to warning: Investigating the presence of text-only/NaN columns 
print(f"Completly empty or text-only-columns:\n{temp_mean}")

Completly empty or text-only-columns:
[54015809.19         nan    15273.46         nan    15311.04         nan       16.62      440.92
         nan         nan         nan         nan         nan     3143.85]


#### Splitting the dataset

In [192]:
## Determining which columns contain text and which contain num-values
str_columns = np.argwhere(np.isnan(temp_mean)).squeeze()
num_columns = np.argwhere(np.isnan(temp_mean) == False).squeeze()

print(f"string columns: {str_columns} \nnumeric columns: {num_columns}")

string columns: [ 1  3  5  8  9 10 11 12] 
numeric columns: [ 0  2  4  6  7 13]


#### Reloading the Dataset with int and string values separate

In [193]:
## Loading string data
loan_data_str = np.genfromtxt("loan-data.csv",
                              delimiter = ';',
                              skip_header = 1,
                              autostrip = True,
                              encoding = 'latin-1',
                              usecols = str_columns,
                              dtype = str)

## Loading numeric data
loan_data_num = np.genfromtxt("loan-data.csv",
                              delimiter = ';',
                              skip_header = 1,
                              autostrip = True,
                              encoding = 'latin-1',
                              usecols = num_columns,
                              filling_values = temp_fill)

print(f"String Dataset:\n{loan_data_str} \n\n\n Numeric Dataset:\n{loan_data_num}")

String Dataset:
[['May-15' 'Current' '36 months' ... 'Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=48010226' 'CA']
 ['' 'Current' '36 months' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=57693261' 'NY']
 ['Sep-15' 'Current' '36 months' ... 'Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=59432726' 'PA']
 ...
 ['Jun-15' 'Current' '36 months' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=50415990' 'CA']
 ['Apr-15' 'Current' '36 months' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=46154151' 'OH']
 ['Dec-15' 'Current' '36 months' ... ''
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=66055249' 'IL']] 


 Numeric Dataset:
[[48010226.      35000.      35000.         13.33     1184.86     9452.96]
 [57693261.      30000.      30000.   68616520.        938.57     4679.7 ]
 [59432726.      15000.      150

#### Naming Columns

In [194]:
header_names = np.genfromtxt("loan-data.csv",
                              delimiter = ';',
                              skip_footer = raw_data.shape[0],
                              autostrip = True,
                              encoding = 'latin-1',
                              dtype = str)
print(f"Header Names: \n {header_names}\n")

header_strings = header_names[str_columns]
header_num = header_names[num_columns]

print(f"String Column Headers:\n{header_strings}\n\nNumeric Colum Headers:\n{header_num}")

Header Names: 
 ['id' 'issue_d' 'loan_amnt' 'loan_status' 'funded_amnt' 'term' 'int_rate' 'installment' 'grade'
 'sub_grade' 'verification_status' 'url' 'addr_state' 'total_pymnt']

String Column Headers:
['issue_d' 'loan_status' 'term' 'grade' 'sub_grade' 'verification_status' 'url' 'addr_state']

Numeric Colum Headers:
['id' 'loan_amnt' 'funded_amnt' 'int_rate' 'installment' 'total_pymnt']


#### Creating a Data Checkpoint

In [195]:
def checkpoint(filename, checkpoint_header, checkpoint_data):
    np.savez(filename, header = checkpoint_header, data = checkpoint_data)
    checkpoint_variable = np.load(filename + ".npz")
    return(checkpoint_variable)

In [196]:
testing_checkpoint = checkpoint("testing_checkpoint", header_strings, loan_data_str)
testing_checkpoint

NpzFile 'testing_checkpoint.npz' with keys: header, data

In [197]:
print(testing_checkpoint['header'],'\n')
print(testing_checkpoint['data'])

['issue_d' 'loan_status' 'term' 'grade' 'sub_grade' 'verification_status' 'url' 'addr_state'] 



[['May-15' 'Current' '36 months' ... 'Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=48010226' 'CA']
 ['' 'Current' '36 months' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=57693261' 'NY']
 ['Sep-15' 'Current' '36 months' ... 'Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=59432726' 'PA']
 ...
 ['Jun-15' 'Current' '36 months' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=50415990' 'CA']
 ['Apr-15' 'Current' '36 months' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=46154151' 'OH']
 ['Dec-15' 'Current' '36 months' ... ''
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=66055249' 'IL']]


#### **Preprocessing String Dataset**

In [198]:
 ## Renaming the column header for clarity
header_strings

array(['issue_d', 'loan_status', 'term', 'grade', 'sub_grade', 'verification_status', 'url',
       'addr_state'], dtype='<U19')

In [199]:
header_strings[0] = "issue_date"
header_strings

array(['issue_date', 'loan_status', 'term', 'grade', 'sub_grade', 'verification_status', 'url',
       'addr_state'], dtype='<U19')

Issue Date

In [200]:
print(loan_data_str[:,0], '\n')

## display unique values
print(np.unique(loan_data_str[:,0]))

['May-15' '' 'Sep-15' ... 'Jun-15' 'Apr-15' 'Dec-15'] 

['' 'Apr-15' 'Aug-15' 'Dec-15' 'Feb-15' 'Jan-15' 'Jul-15' 'Jun-15' 'Mar-15' 'May-15' 'Nov-15'
 'Oct-15' 'Sep-15']


In [201]:
## Removing the '-15'
loan_data_str[:,0] = np.chararray.strip(loan_data_str[:,0], "-15")

np.unique(loan_data_str[:,0])

C:\Users\Bhrajo\AppData\Local\Temp\ipykernel_12628\2786919861.py:2: DeprecationWarning: `np.chararray` is deprecated and will be removed from the main namespace in the future. Use an array with a string or bytes dtype instead.
  loan_data_str[:,0] = np.chararray.strip(loan_data_str[:,0], "-15")


array(['', 'Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep'],
      dtype='<U69')

In [202]:
months = np.array(['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

for i in range(13):
    loan_data_str[:,0] = np.where(loan_data_str[:,0] == months[i],
                                  str(i), 
                                  loan_data_str[:,0])

print(np.unique(loan_data_str[:,0]))

['0' '1' '10' '11' '12' '2' '3' '4' '5' '6' '7' '8' '9']


Loan Status

In [203]:
header_strings

array(['issue_date', 'loan_status', 'term', 'grade', 'sub_grade', 'verification_status', 'url',
       'addr_state'], dtype='<U19')

In [204]:
## Checking the number of unique element in the "loan status" column
print(f"Total number of unique elements in 'loan status column': {np.unique(loan_data_str[:,1]).size}\n")
print(np.unique(loan_data_str[:,1]))

Total number of unique elements in 'loan status column': 9

['' 'Charged Off' 'Current' 'Default' 'Fully Paid' 'In Grace Period' 'Issued' 'Late (16-30 days)'
 'Late (31-120 days)']


In [205]:
## Classifying the 'loan status' into good (1) and bad (0)
bad_loan_status = np.array(['','Charged Off', 'Default', 'Late (31-120 days)'])
loan_data_str[:,1] = np.where(np.isin(loan_data_str[:,1], bad_loan_status),0,1)

np.unique(loan_data_str[:,1])

array(['0', '1'], dtype='<U69')

Loan Term

In [206]:
print(header_strings,'\n')
print(np.unique(loan_data_str[:,2]))
print(f"Total unique values in term: {np.unique(loan_data_str[:,2]).size}")

['issue_date' 'loan_status' 'term' 'grade' 'sub_grade' 'verification_status' 'url' 'addr_state'] 

['' '36 months' '60 months']
Total unique values in term: 3


In [207]:
loan_data_str[:,2] = np.chararray.strip(loan_data_str[:,2], " months")
header_strings[2] = "term_months"
loan_data_str[:,2]

C:\Users\Bhrajo\AppData\Local\Temp\ipykernel_12628\703178168.py:1: DeprecationWarning: `np.chararray` is deprecated and will be removed from the main namespace in the future. Use an array with a string or bytes dtype instead.
  loan_data_str[:,2] = np.chararray.strip(loan_data_str[:,2], " months")


array(['36', '36', '36', ..., '36', '36', '36'], shape=(10000,), dtype='<U69')

In [208]:
loan_data_str[:,2] =  np.where(loan_data_str[:,2] == '', '60', loan_data_str[:,2])
print(loan_data_str[:,2],'\n')
print(np.unique(loan_data_str[:,2]))

['36' '36' '36' ... '36' '36' '36'] 

['36' '60']


grade and subgrade

In [209]:
print(header_strings, '\n')
print(f"Unique values in 'grade' column:\n{np.unique(loan_data_str[:,3])}\n")
print(f"Unique values in 'sub_grade' column:\n{np.unique(loan_data_str[:,4])}")

['issue_date' 'loan_status' 'term_months' 'grade' 'sub_grade' 'verification_status' 'url'
 'addr_state'] 

Unique values in 'grade' column:
['' 'A' 'B' 'C' 'D' 'E' 'F' 'G']

Unique values in 'sub_grade' column:
['' 'A1' 'A2' 'A3' 'A4' 'A5' 'B1' 'B2' 'B3' 'B4' 'B5' 'C1' 'C2' 'C3' 'C4' 'C5' 'D1' 'D2' 'D3' 'D4'
 'D5' 'E1' 'E2' 'E3' 'E4' 'E5' 'F1' 'F2' 'F3' 'F4' 'F5' 'G1' 'G2' 'G3' 'G4' 'G5']


In [ ]:
## Handling missing values in sub_grade column
for i in np.unique(loan_data_str[:,3])[1]:
                   loan_data_str[:,4] = np.where((loan_data_str[:,4] == '') & 
                                                 (loan_data_str[:,3] == 1),
                   i + '5', loan_data_str[:,4])

np.unique(loan_data_str[:,4], return_counts = True)

(array(['', 'A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'C4',
        'C5', 'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2', 'E3', 'E4', 'E5', 'F1', 'F2', 'F3', 'F4',
        'F5', 'G1', 'G2', 'G3', 'G4', 'G5'], dtype='<U69'),
 array([514, 285, 278, 239, 323, 502, 509, 517, 530, 553, 494, 629, 567, 586, 564, 423, 391, 267,
        250, 255, 223, 235, 162, 171, 139, 114,  94,  52,  34,  43,  16,  19,  10,   3,   7,   2]))

In [211]:
loan_data_str[:,4] = np.where((loan_data_str[:,4] ==''), 'H1', loan_data_str[:,4])

np.unique(loan_data_str[:,4])

array(['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'C4', 'C5',
       'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2', 'E3', 'E4', 'E5', 'F1', 'F2', 'F3', 'F4', 'F5',
       'G1', 'G2', 'G3', 'G4', 'G5', 'H1'], dtype='<U69')

In [212]:
## Removing the grade column because its the same as the sub-grade column
loan_data_str = np.delete(loan_data_str, 3, axis =  1)

header_strings = np.delete(header_strings, 3)

print(f"New Data:{loan_data_str}\n\nNew Header_strings: {header_strings}")

New Data:[['5' '1' '36' ... 'Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=48010226' 'CA']
 ['0' '1' '36' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=57693261' 'NY']
 ['9' '1' '36' ... 'Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=59432726' 'PA']
 ...
 ['6' '1' '36' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=50415990' 'CA']
 ['4' '1' '36' ... 'Source Verified'
  'https://www.lendingclub.com/browse/loanDetail.action?loan_id=46154151' 'OH']
 ['12' '1' '36' ... '' 'https://www.lendingclub.com/browse/loanDetail.action?loan_id=66055249'
  'IL']]

New Header_strings: ['issue_date' 'loan_status' 'term_months' 'sub_grade' 'verification_status' 'url' 'addr_state']


Converting Sub Grade 

In [213]:
np.unique(loan_data_str[:,3])

array(['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'C4', 'C5',
       'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2', 'E3', 'E4', 'E5', 'F1', 'F2', 'F3', 'F4', 'F5',
       'G1', 'G2', 'G3', 'G4', 'G5', 'H1'], dtype='<U69')

Verification Status

In [214]:
header_strings

array(['issue_date', 'loan_status', 'term_months', 'sub_grade', 'verification_status', 'url',
       'addr_state'], dtype='<U19')

In [216]:
np.unique(loan_data_str[:,4])

array(['', 'Not Verified', 'Source Verified', 'Verified'], dtype='<U69')

In [235]:
loan_data_str[:,4] = np.where((loan_data_str[:,4] == '') | (loan_data_str[:,4] == 'Not Verified'), 0, 1)

In [238]:
np.unique(loan_data_str[:,4])

array(['1'], dtype='<U69')

Url

In [239]:
loan_data_str[:,5]

array(['https://www.lendingclub.com/browse/loanDetail.action?loan_id=48010226',
       'https://www.lendingclub.com/browse/loanDetail.action?loan_id=57693261',
       'https://www.lendingclub.com/browse/loanDetail.action?loan_id=59432726', ...,
       'https://www.lendingclub.com/browse/loanDetail.action?loan_id=50415990',
       'https://www.lendingclub.com/browse/loanDetail.action?loan_id=46154151',
       'https://www.lendingclub.com/browse/loanDetail.action?loan_id=66055249'],
      shape=(10000,), dtype='<U69')

In [240]:
np.chararray.strip(loan_data_str[:,5], "https://www.lendingclub.com/browse/loanDetail.action?loan_id=")

C:\Users\Bhrajo\AppData\Local\Temp\ipykernel_12628\820869075.py:1: DeprecationWarning: `np.chararray` is deprecated and will be removed from the main namespace in the future. Use an array with a string or bytes dtype instead.
  np.chararray.strip(loan_data_str[:,5], "https://www.lendingclub.com/browse/loanDetail.action?loan_id=")


array(['48010226', '57693261', '59432726', ..., '50415990', '46154151', '66055249'],
      shape=(10000,), dtype='<U69')

In [241]:
loan_data_str[:,5] = np.chararray.strip(loan_data_str[:,5], "https://www.lendingclub.com/browse/loanDetail.action?loan_id=")
loan_data_str[:,5]

C:\Users\Bhrajo\AppData\Local\Temp\ipykernel_12628\3759624371.py:1: DeprecationWarning: `np.chararray` is deprecated and will be removed from the main namespace in the future. Use an array with a string or bytes dtype instead.
  loan_data_str[:,5] = np.chararray.strip(loan_data_str[:,5], "https://www.lendingclub.com/browse/loanDetail.action?loan_id=")


array(['48010226', '57693261', '59432726', ..., '50415990', '46154151', '66055249'],
      shape=(10000,), dtype='<U69')